# ReupStudio Colab One-Click Notebook

Notebook này giúp bạn khởi chạy dự án Auto-Reup Video Studio trên Google Colab chỉ với 3 bước đơn giản.

## Cell 1: Kích hoạt GPU & Clone Repo

Chạy cell này để kiểm tra GPU của Colab trước khi clone repository. Nếu GPU chưa bật, notebook sẽ dừng và yêu cầu bạn bật Runtime GPU ở Colab.

In [ ]:
# Kiểm tra GPU trước khi tiếp tục
import os
import subprocess
import sys

try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("GPU không khả dụng. Vui lòng bật GPU trong Colab: Runtime -> Change runtime type -> GPU")
    print("GPU OK:", torch.cuda.get_device_name(0))
except Exception as exc:
    raise RuntimeError(f"GPU check failed: {exc}") from exc

# Clone repository từ GitHub
!git clone https://github.com/11chucuncon/ReupAllInOne.git

# Chuyển vào thư mục dự án
%cd ReupAllInOne

# Cập nhật code mới nhất từ GitHub
!git pull

## Cell 2: Cài đặt Hệ thống & Thư viện

Cell này cài FFmpeg cho Linux Colab và cài toàn bộ dependencies Python từ requirements.txt.

In [ ]:
# Cài FFmpeg cho Linux Colab
!apt-get update -y && apt-get install ffmpeg -y

# Cài font hỗ trợ tiếng Việt và các ngôn ngữ CJK cho subtitles
!apt-get install -y fonts-noto-cjk fonts-noto-core fonts-dejavu-core
!fc-cache -f -v

# Cài đặt thư viện Python cơ bản cho GPU và audio
!pip install torch torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

# Cài đặt các dependency khác từ requirements
!pip install -r requirements.txt

# Sử dụng Edge-TTS cho Python 3.12 vì ổn định và nhẹ
!pip install edge-tts

# Cài đặt Real-ESRGAN và các gói liên quan
!pip install git+https://github.com/xinntao/Real-ESRGAN.git
!pip install gfpgan
!pip install --force-reinstall --no-deps basicsr

# Patch lỗi torchvision.transforms.functional_tensor trong basicsr an toàn bằng sed
!sed -i 's/functional_tensor/functional/g' /usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py

# Clone và cài ProPainter để kích hoạt AI inpainting
!rm -rf core/ProPainter
!git clone https://github.com/sczhou/ProPainter.git core/ProPainter
!pip install -r core/ProPainter/requirements.txt
!mkdir -p core/ProPainter/weights
!wget -nc https://github.com/sczhou/ProPainter/releases/download/v0.1.0/ProPainter.pth -P core/ProPainter/weights/
!wget -nc https://github.com/sczhou/ProPainter/releases/download/v0.1.0/raft-things.pth -P core/ProPainter/weights/
!wget -nc https://github.com/sczhou/ProPainter/releases/download/v0.1.0/recurrent_flow_completion.pth -P core/ProPainter/weights/

!mkdir -p weights
!wget -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth -P weights/


## Cell 3: Khởi chạy Web UI Gradio

Cell này sẽ chạy giao diện Gradio ở chế độ public share để bạn mở trực tiếp trên Colab.

In [ ]:
# Khởi chạy ứng dụng Gradio trên Colab
!python run_colab.py